# 15 — Artificial neural-network candidate screen

This notebook presents the frozen-fold evidence produced by the bounded MLP screen. The labelled local test remains unopened; no oversampling or competition prediction is involved.

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd

project_dir = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'stage-1-pump-it-up').exists()
)
stage_dir = project_dir / 'stage-1-pump-it-up'
src_dir = stage_dir / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
screen = joblib.load(project_dir / '.runtime' / 'ann-screen' / 'ann-screen.joblib')
baseline = screen['baseline']
print(f"Frozen-fold baseline: {baseline.metric_summary.loc['accuracy', 'mean']:.4%}")
print(f"ANN candidates: {len(screen['candidates'])}; fixed blends: {len(screen['blends'])}")

Frozen-fold baseline: 81.6246%
ANN candidates: 7; fixed blends: 84


## Standalone MLPs

Numeric values are standardised inside each training fold. Sparse one-hot categories use the existing rare-category policy. Adam training uses a fixed seed and a training-fold-only 10% early-stopping split.

In [2]:
standalone = screen['standalone_summary'].copy()
standalone[['model_name', 'mean_accuracy', 'accuracy_change', 'repair_recall', 'mean_iterations']].style.format({
    'mean_accuracy': '{:.3%}',
    'accuracy_change': '{:+.3%}',
    'repair_recall': '{:.3%}',
    'mean_iterations': '{:.1f}',
})

,model_name,mean_accuracy,accuracy_change,repair_recall,mean_iterations
candidate,,,,,
relu_128_64,MLP ReLU 128-64 [scaled current one-hot],78.590%,-3.035%,24.203%,35.0
relu_128_64_funder_identity_frequency,MLP ReLU 128-64 [scaled funder rare20 frequency],78.569%,-3.056%,26.260%,28.0
relu_128_64_regularised,MLP ReLU 128-64 regularised [scaled current one-hot],78.371%,-3.253%,24.087%,27.8
relu_128_64_both_identity_frequency,MLP ReLU 128-64 [scaled both rare20 frequency],78.312%,-3.312%,28.374%,27.6
relu_128,MLP ReLU 128 [scaled current one-hot],78.270%,-3.354%,24.725%,32.2
relu_128_64_funder_frequency,MLP ReLU 128-64 [scaled funder frequency],78.232%,-3.392%,27.360%,31.6
relu_64,MLP ReLU 64 [scaled current one-hot],78.197%,-3.428%,24.638%,53.6


## Fixed probability blends

Each candidate receives one of twelve pre-declared weights from 1% to 50%. The accepted ensemble supplies the remainder.

In [3]:
blends = screen['blend_summary'].copy()
print(f"Gate-passing blends: {int(blends['passes_gate'].sum())}")
blends.head(12)[['model_name', 'mean_accuracy', 'accuracy_change', 'fold_wins', 'worst_fold_change', 'repair_recall', 'passes_gate']].style.format({
    'mean_accuracy': '{:.3%}',
    'accuracy_change': '{:+.3%}',
    'worst_fold_change': '{:+.3%}',
    'repair_recall': '{:.3%}',
})

Gate-passing blends: 0


,,model_name,mean_accuracy,accuracy_change,fold_wins,worst_fold_change,repair_recall,passes_gate
candidate,ann_weight,,,,,,,
relu_64,0.020000,98% accepted ensemble + 2% MLP ReLU 64 [scaled current one-hot],81.616%,-0.008%,1,-0.074%,34.801%,False
relu_128_64_funder_frequency,0.020000,98% accepted ensemble + 2% MLP ReLU 128-64 [scaled funder frequency],81.612%,-0.013%,2,-0.116%,34.801%,False
relu_128_64_regularised,0.010000,99% accepted ensemble + 1% MLP ReLU 128-64 regularised [scaled current one-hot],81.610%,-0.015%,2,-0.053%,34.772%,False
relu_64,0.010000,99% accepted ensemble + 1% MLP ReLU 64 [scaled current one-hot],81.610%,-0.015%,1,-0.063%,34.801%,False
relu_128_64,0.040000,96% accepted ensemble + 4% MLP ReLU 128-64 [scaled current one-hot],81.608%,-0.017%,2,-0.116%,34.627%,False
relu_128_64_both_identity_frequency,0.020000,98% accepted ensemble + 2% MLP ReLU 128-64 [scaled both rare20 frequency],81.608%,-0.017%,1,-0.084%,34.859%,False
relu_128_64_regularised,0.020000,98% accepted ensemble + 2% MLP ReLU 128-64 regularised [scaled current one-hot],81.608%,-0.017%,2,-0.116%,34.685%,False
relu_128_64_both_identity_frequency,0.010000,99% accepted ensemble + 1% MLP ReLU 128-64 [scaled both rare20 frequency],81.606%,-0.019%,2,-0.053%,34.830%,False
relu_128_64_funder_frequency,0.010000,99% accepted ensemble + 1% MLP ReLU 128-64 [scaled funder frequency],81.606%,-0.019%,2,-0.084%,34.743%,False


## Error diversity relative to the accepted vote

In [4]:
diversity = screen['diversity'].reset_index()
diversity = diversity[diversity['left_model'] == baseline.model_name].set_index('right_model')
diversity[['disagreement', 'left_only_correct', 'right_only_correct', 'both_wrong']].style.format('{:.3%}')

,disagreement,left_only_correct,right_only_correct,both_wrong
right_model,,,,
MLP ReLU 64 [scaled current one-hot],11.425%,6.936%,3.508%,14.867%
MLP ReLU 128 [scaled current one-hot],11.444%,6.953%,3.598%,14.777%
MLP ReLU 128-64 [scaled current one-hot],10.871%,6.475%,3.441%,14.935%
MLP ReLU 128-64 regularised [scaled current one-hot],11.362%,6.833%,3.580%,14.796%
MLP ReLU 128-64 [scaled funder frequency],11.625%,6.999%,3.607%,14.769%
MLP ReLU 128-64 [scaled funder rare20 frequency],11.465%,6.715%,3.660%,14.716%
MLP ReLU 128-64 [scaled both rare20 frequency],11.742%,7.033%,3.721%,14.655%


## Decision

Retain the accepted XGBoost/Random Forest vote. The strongest MLP reaches 78.590% standalone; the closest blend reaches 81.616%, still below the 81.625% baseline. Stop architecture, seed and blend-weight tuning on these folds. A future neural revisit would need a materially different representation such as learned embeddings and an explicitly justified framework, not a larger search over the same one-hot MLP.